In [8]:
from playerPredictor import identify_nba_season, calculate_recent_scores, calculate_player_metrics
from playerPredictor import get_team_player_scores
from teamPredictor import calculate_recent_form_scores, calculate_net_four
from teamPredictor import get_historical_matchups#, calculate_historic_matchups
#identify_nba_season('12/22/2020')

In [10]:
def compute_team_strength_score(engine, date, team_name):

    #compute team performance scores
    recent_form = calculate_recent_form_scores(engine, date, team_name)
    season_net, season_four_factors = calculate_net_four(engine, date, team_name)
    
    team_performance = (0.7 * recent_form) + (0.3 * (0.5 * season_net + 0.5 * season_four_factors))
    
    #computer player contribution score
    player_scores_df = get_team_player_scores(engine, team_name, date)
    
    star_players = player_scores_df[player_scores_df['CATEGORY'] == 'Star']
    rotation_players = player_scores_df[player_scores_df['CATEGORY'] == 'Rotation']
    
    star_score = star_players['SCORE'].mean()
    rotation_score = rotation_players['SCORE'].mean()
    
    player_contribution = (0.7 * star_score) + (0.3 * rotation_score)
    
    #Computer Historical Matchup Scores
    #historical_matchups = calculate_historic_matchups(engine, date, team_name, opponent_team, avg_games=3, weighted=True)
    
    # historical_matchup_score = (0.5 * historical_matchups['net_rating_diff']) + \
    #                            (0.3 * historical_matchups['efg_diff']) + \
    #                            (0.2 * historical_matchups['turnover_pct_diff'])
    
    TSS = (0.5 * team_performance) + (0.5 * player_contribution)# + (0.15 * historical_matchup_score)
    return TSS

In [12]:
from datetime import datetime
from sqlalchemy import create_engine, text
import pandas as pd
import math
from dotenv import load_dotenv
import os

load_dotenv()

DATABASE_URL = os.getenv("DATABASE_URL")

engine = create_engine(DATABASE_URL)
team_name = 'Detroit Pistons'
#date = '03-13-2023'
date = '2024-03-13'
compute_team_strength_score(engine, date, team_name)

Net Rating Score: -0.6839999999999994
Recent Four Factors Score: 0.51343


C:\Users\Chase\ML Algorithm\JupyterScrapers\predictionScripts\teamPredictor.py:364: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  games_df = pd.concat([games_df, individual_team_df], ignore_index=True)


Net Rating Score: -7.840625000000001
Recent Four Factors Score: -3.2794597596153845


DataError: (psycopg2.errors.InvalidDatetimeFormat) invalid value "2025-02-03" for "MON"
DETAIL:  The given value did not match any of the allowed values for this field.

[SQL: 
                WITH recent_games AS (
                    SELECT *
                    FROM "all_player_game_stats_2024-25"
                    WHERE "PLAYER_ID" = %(player_id)s
                    AND TO_DATE("GAME_DATE", 'MON DD, YYYY') < TO_DATE(%(date)s, 'YYYY-MM-DD')
                    ORDER BY TO_DATE("GAME_DATE", 'MON DD, YYYY') DESC
                    LIMIT 10
                )
                SELECT
                    "PLAYER_ID",
                    AVG("PTS") as "AVG_PTS",
                    AVG("PLUS_MINUS") as "AVG_PLUS_MINUS",
                    AVG("MIN") as "AVG_MIN",
                    AVG("FGA") as "AVG_FGA", 
                    AVG("FTA") as "AVG_FTA",
                    AVG("TOV") as "AVG_TOV",
                    AVG("OREB") as "AVG_OREB",
                    MAX(TO_DATE("GAME_DATE"

In [37]:
from playerPredictor import identify_nba_season, calculate_recent_scores, calculate_player_metrics
from playerPredictor import get_team_player_scores
from teamPredictor import calculate_recent_form_scores, calculate_net_four
from teamPredictor import get_historical_matchups
identify_nba_season('12/22/2020')

'2020-21'